**Smart Education Analytics and Student Performance Prediction System using Apache Spark**

Q1. Spark Initialization and Data Loading

In [42]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

# Initialize Spark Session
spark = SparkSession.builder \
    .appName("Smart Education Analytics and Student Performance Prediction") \
    .master("local[*]") \
    .getOrCreate()

sc = spark.sparkContext

# Verify Spark
print("Spark Version:", spark.version)
print("Application Name:", sc.appName)
print("Master:", sc.master)

# Load datasets
student_mat_df = spark.read.csv(
    "/content/student-mat.csv",
    header=True,
    inferSchema=True,
    sep=";"
)

student_por_df = spark.read.csv(
    "/content/student-por.csv",
    header=True,
    inferSchema=True,
    sep=";"
)

placement_df = spark.read.csv(
    "/content/Placement_Data_Full_Class.csv",
    header=True,
    inferSchema=True
)

xapi_df = spark.read.csv(
    "/content/xAPI-Edu-Data.csv",
    header=True,
    inferSchema=True
)

Spark Version: 4.0.3
Application Name: Smart Education Analytics and Student Performance Prediction
Master: local[*]


In [43]:
#Checking Loaded Data
print("Math Student Data")
student_mat_df.show(5)

print("Portuguese Student Data")
student_por_df.show(5)

print("Placement Data")
placement_df.show(5)

print("xAPI Education Data")
xapi_df.show(5)

Math Student Data
+------+---+---+-------+-------+-------+----+----+-------+--------+------+--------+----------+---------+--------+---------+------+----+----------+-------+------+--------+--------+------+--------+-----+----+----+------+--------+---+---+---+
|school|sex|age|address|famsize|Pstatus|Medu|Fedu|   Mjob|    Fjob|reason|guardian|traveltime|studytime|failures|schoolsup|famsup|paid|activities|nursery|higher|internet|romantic|famrel|freetime|goout|Dalc|Walc|health|absences| G1| G2| G3|
+------+---+---+-------+-------+-------+----+----+-------+--------+------+--------+----------+---------+--------+---------+------+----+----------+-------+------+--------+--------+------+--------+-----+----+----+------+--------+---+---+---+
|    GP|  F| 18|      U|    GT3|      A|   4|   4|at_home| teacher|course|  mother|         2|        2|       0|      yes|    no|  no|        no|    yes|   yes|      no|      no|     4|       3|    4|   1|   1|     3|       6|  5|  6|  6|
|    GP|  F| 17|      

In [44]:
#Checking Schema
student_mat_df.printSchema()
student_por_df.printSchema()
placement_df.printSchema()
xapi_df.printSchema()

root
 |-- school: string (nullable = true)
 |-- sex: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- address: string (nullable = true)
 |-- famsize: string (nullable = true)
 |-- Pstatus: string (nullable = true)
 |-- Medu: integer (nullable = true)
 |-- Fedu: integer (nullable = true)
 |-- Mjob: string (nullable = true)
 |-- Fjob: string (nullable = true)
 |-- reason: string (nullable = true)
 |-- guardian: string (nullable = true)
 |-- traveltime: integer (nullable = true)
 |-- studytime: integer (nullable = true)
 |-- failures: integer (nullable = true)
 |-- schoolsup: string (nullable = true)
 |-- famsup: string (nullable = true)
 |-- paid: string (nullable = true)
 |-- activities: string (nullable = true)
 |-- nursery: string (nullable = true)
 |-- higher: string (nullable = true)
 |-- internet: string (nullable = true)
 |-- romantic: string (nullable = true)
 |-- famrel: integer (nullable = true)
 |-- freetime: integer (nullable = true)
 |-- goout: integer (null

In [45]:
#Count Records
print("Math Dataset Records:", student_mat_df.count())
print("Portuguese Dataset Records:", student_por_df.count())
print("Placement Dataset Records:", placement_df.count())
print("xAPI Dataset Records:", xapi_df.count())

Math Dataset Records: 395
Portuguese Dataset Records: 649
Placement Dataset Records: 215
xAPI Dataset Records: 480


Spark is successfully initialized and all four educational datasets are loaded into Spark DataFrames. These datasets cover academic scores, attendance, online learning activity, and placement records, so they are suitable for the full Smart Education Analytics case study.

**Q2. RDD Implementation**

In [5]:
#Create RDDs and perform transformation and action operations.
# Create RDDs
mat_rdd = student_mat_df.rdd
por_rdd = student_por_df.rdd
xapi_rdd = xapi_df.rdd
placement_rdd = placement_df.rdd
#DataFrames are converted into RDDs so we can perform low-level distributed operations.

# Actions
print("Math RDD Count:", mat_rdd.count())
print("Portuguese RDD Count:", por_rdd.count())
print("xAPI RDD Count:", xapi_rdd.count())
print("Placement RDD Count:", placement_rdd.count())
#count() is an action that returns the total number of records in each RDD.

print("First 5 Math Records:")
print(mat_rdd.take(5))

# Transformations
high_score_students = mat_rdd.filter(lambda row: row["G3"] >= 15)
print("High Score Students:")
print(high_score_students.take(5))
#filter() is a transformation.It selects students who scored high marks in the final exam.

student_marks = mat_rdd.map(lambda row: (row["sex"], row["age"], row["G1"], row["G2"], row["G3"]))
print("Selected Marks:")
print(student_marks.take(5))
#map() is a transformation.It extracts only important academic columns from the dataset.

total_marks_rdd = mat_rdd.map(lambda row: (row["sex"], row["G1"] + row["G2"] + row["G3"]))
print("Total Marks:")
print(total_marks_rdd.take(5))
#This calculates total academic performance using three exam scores: G1, G2, and G3.

low_attendance_rdd = mat_rdd.filter(lambda row: row["absences"] > 10)
print("Low Attendance Students:")
print(low_attendance_rdd.take(5))
#Students with more than 10 absences are identified as low-attendance students.

online_activity_rdd = xapi_rdd.map(
    lambda row: (
        row["gender"],
        row["raisedhands"],
        row["VisITedResources"],
        row["AnnouncementsView"],
        row["Discussion"],
        row["Class"]
    )
)
print("Online Activity Sample:")
print(online_activity_rdd.take(5))
#This extracts online learning activity columns such as raised hands, visited resources, announcements viewed, and discussions.

performance_class_count = xapi_rdd.map(lambda row: row["Class"]).countByValue()
print("Performance Class Count:", performance_class_count)
#countByValue() is an action.It counts students in each performance category such as High, Medium, and Low.


placement_status_count = placement_rdd.map(lambda row: row["status"]).countByValue()
print("Placement Status Count:", placement_status_count)
#This counts how many students were placed and not placed.

max_final_score = mat_rdd.map(lambda row: row["G3"]).max()
print("Maximum Final Score:", max_final_score)

avg_final_score = mat_rdd.map(lambda row: row["G3"]).mean()
print("Average Final Score:", avg_final_score)

Math RDD Count: 395
Portuguese RDD Count: 649
xAPI RDD Count: 480
Placement RDD Count: 215
First 5 Math Records:
[Row(school='GP', sex='F', age=18, address='U', famsize='GT3', Pstatus='A', Medu=4, Fedu=4, Mjob='at_home', Fjob='teacher', reason='course', guardian='mother', traveltime=2, studytime=2, failures=0, schoolsup='yes', famsup='no', paid='no', activities='no', nursery='yes', higher='yes', internet='no', romantic='no', famrel=4, freetime=3, goout=4, Dalc=1, Walc=1, health=3, absences=6, G1=5, G2=6, G3=6), Row(school='GP', sex='F', age=17, address='U', famsize='GT3', Pstatus='T', Medu=1, Fedu=1, Mjob='at_home', Fjob='other', reason='course', guardian='father', traveltime=1, studytime=2, failures=0, schoolsup='no', famsup='yes', paid='no', activities='no', nursery='no', higher='yes', internet='yes', romantic='no', famrel=5, freetime=3, goout=3, Dalc=1, Walc=1, health=3, absences=4, G1=5, G2=5, G3=6), Row(school='GP', sex='F', age=15, address='U', famsize='LE3', Pstatus='T', Medu=1,

RDDs were created from the educational datasets and both transformation and action operations were performed.
Transformations used: map(), filter()

Actions used:
take()
count()
countByValue()
max()
mean()

RDD operations helped analyze student marks, attendance, online activity, performance class, and placement status.

**Q3. Key-Value Operations and Persistence**

In [6]:
#Implement key-value operations, shuffle operations, and persistence
performance_kv_rdd = xapi_rdd.map(lambda row: (row["Class"], 1))
print(performance_kv_rdd.take(5))
#Each student is converted into a key-value pair: (Class, 1) This helps count students in each performance category.

performance_count = performance_kv_rdd.reduceByKey(lambda a, b: a + b)
print(performance_count.collect())
#reduceByKey() groups records with the same key and adds their values.This gives count of students in each class:High, Medium, Low

gender_score_rdd = mat_rdd.map(
    lambda row: (row["sex"], (row["G3"], 1))
)

gender_avg_score = gender_score_rdd.reduceByKey(
    lambda a, b: (a[0] + b[0], a[1] + b[1])
).mapValues(
    lambda x: x[0] / x[1]
)

print(gender_avg_score.collect())
#This calculates average final exam score based on gender.

gender_group_scores = mat_rdd.map(
    lambda row: (row["sex"], row["G3"])
).groupByKey()

for gender, scores in gender_group_scores.collect():
    print(gender, list(scores)[:10])
#groupByKey() groups all marks under the same gender.But it causes more shuffle, so it is slower than reduceByKey() for aggregation.

sorted_performance = performance_count.sortByKey()
print(sorted_performance.collect())
#sortByKey() sorts performance categories alphabetically.Sorting requires data movement between partitions, so it is also a shuffle operation.

placement_kv_rdd = placement_rdd.map(
    lambda row: (row["status"], 1))
placement_count = placement_kv_rdd.reduceByKey(
    lambda a, b: a + b )
print(placement_count.collect())
#This calculates number of placed and not placed students.

academic_rdd = mat_rdd.map(
    lambda row: (
        row["sex"],
        row["age"],
        row["absences"],
        row["G1"],
        row["G2"],
        row["G3"]
    )
)

academic_rdd.cache()
#cache() stores the RDD in memory.This is useful because the same academic RDD is reused multiple times.

print("Total Academic Records:", academic_rdd.count())
high_performers = academic_rdd.filter(lambda row: row[5] >= 15)
print("High Performers:", high_performers.count())
low_attendance = academic_rdd.filter(lambda row: row[2] > 10)
print("Low Attendance Students:", low_attendance.count())
#Since academic_rdd is cached, Spark does not recompute it every time.This improves performance.

academic_rdd.unpersist()
#unpersist() removes cached data from memory after use.This prevents memory wastage.

[('M', 1), ('M', 1), ('L', 1), ('L', 1), ('M', 1)]
[('M', 211), ('L', 127), ('H', 142)]
[('F', 9.966346153846153), ('M', 10.914438502673796)]
F [6, 6, 10, 15, 10, 6, 9, 12, 14, 14]
M [15, 11, 19, 15, 14, 11, 16, 5, 10, 15]
[('H', 142), ('L', 127), ('M', 211)]
[('Placed', 148), ('Not Placed', 67)]
Total Academic Records: 395
High Performers: 73
Low Attendance Students: 66


PythonRDD[140] at RDD at PythonRDD.scala:56

In this section, key-value RDD operations were implemented using:

map()
reduceByKey()
mapValues()
groupByKey()
sortByKey()
cache()
unpersist()

reduceByKey() is preferred over groupByKey() for aggregation because it reduces data before shuffle and improves performance.

**Q4. Spark DataFrame Operations**

In [7]:
#Perform DataFrame operations including joins and aggregation.
from pyspark.sql.functions import monotonically_increasing_id

student_mat_df = student_mat_df.withColumn(
    "Student_ID",
    monotonically_increasing_id()
)

student_por_df = student_por_df.withColumn(
    "Student_ID",
    monotonically_increasing_id()
)

placement_df = placement_df.withColumn(
    "Student_ID",
    monotonically_increasing_id()
)

xapi_df = xapi_df.withColumn(
    "Student_ID",
    monotonically_increasing_id()
)
#Since the datasets do not share a common primary key, a synthetic Student_ID is created for demonstration purposes. This enables DataFrame joins required by the case study.

In [8]:
academic_df = student_mat_df.join(
    xapi_df,
    on="Student_ID",
    how="inner"
)

academic_df.show(5)

+----------+------+---+---+-------+-------+-------+----+----+-------+--------+------+--------+----------+---------+--------+---------+------+----+----------+-------+------+--------+--------+------+--------+-----+----+----+------+--------+---+---+---+------+-----------+------------+----------+-------+---------+-----+--------+--------+-----------+----------------+-----------------+----------+---------------------+------------------------+------------------+-----+
|Student_ID|school|sex|age|address|famsize|Pstatus|Medu|Fedu|   Mjob|    Fjob|reason|guardian|traveltime|studytime|failures|schoolsup|famsup|paid|activities|nursery|higher|internet|romantic|famrel|freetime|goout|Dalc|Walc|health|absences| G1| G2| G3|gender|NationalITy|PlaceofBirth|   StageID|GradeID|SectionID|Topic|Semester|Relation|raisedhands|VisITedResources|AnnouncementsView|Discussion|ParentAnsweringSurvey|ParentschoolSatisfaction|StudentAbsenceDays|Class|
+----------+------+---+---+-------+-------+-------+----+----+-------

The academic dataset is joined with the xAPI dataset using Student_ID. The joined DataFrame combines exam scores with attendance and online learning activities.

In [9]:
education_df = academic_df.join(
    placement_df,
    on="Student_ID",
    how="left"
)
education_df.show(5)

+----------+------+---+---+-------+-------+-------+----+----+-------+--------+------+--------+----------+---------+--------+---------+------+----+----------+-------+------+--------+--------+------+--------+-----+----+----+------+--------+---+---+---+------+-----------+------------+----------+-------+---------+-----+--------+--------+-----------+----------------+-----------------+----------+---------------------+------------------------+------------------+-----+-----+------+-----+-------+-----+-------+--------+--------+---------+------+-------+--------------+-----+----------+------+
|Student_ID|school|sex|age|address|famsize|Pstatus|Medu|Fedu|   Mjob|    Fjob|reason|guardian|traveltime|studytime|failures|schoolsup|famsup|paid|activities|nursery|higher|internet|romantic|famrel|freetime|goout|Dalc|Walc|health|absences| G1| G2| G3|gender|NationalITy|PlaceofBirth|   StageID|GradeID|SectionID|Topic|Semester|Relation|raisedhands|VisITedResources|AnnouncementsView|Discussion|ParentAnsweringSur

A left join is performed to combine academic, online activity, and placement information into one DataFrame.

In [10]:
education_df.select(
    "Student_ID",
    "sex",
    "age",
    "G3",
    "raisedhands",
    "VisITedResources",
    "AnnouncementsView",
    "Discussion",
    "status"
).show(10)

+----------+---+---+---+-----------+----------------+-----------------+----------+----------+
|Student_ID|sex|age| G3|raisedhands|VisITedResources|AnnouncementsView|Discussion|    status|
+----------+---+---+---+-----------+----------------+-----------------+----------+----------+
|         0|  F| 18|  6|         15|              16|                2|        20|    Placed|
|         1|  F| 17|  6|         20|              20|                3|        25|    Placed|
|         2|  F| 15| 10|         10|               7|                0|        30|    Placed|
|         3|  F| 15| 15|         30|              25|                5|        35|Not Placed|
|         4|  F| 16| 10|         40|              50|               12|        50|    Placed|
|         5|  M| 16| 15|         42|              30|               13|        70|Not Placed|
|         6|  M| 16| 11|         35|              12|                0|        17|Not Placed|
|         7|  F| 17|  6|         50|              10|       

Only the important columns required for educational analytics are selected, reducing unnecessary data processing.

In [11]:
high_performers = education_df.filter(education_df.G3 >= 15 )
high_performers.show()


+----------+------+---+---+-------+-------+-------+----+----+--------+--------+----------+--------+----------+---------+--------+---------+------+----+----------+-------+------+--------+--------+------+--------+-----+----+----+------+--------+---+---+---+------+-----------+------------+------------+-------+---------+-------+--------+--------+-----------+----------------+-----------------+----------+---------------------+------------------------+------------------+-----+-----+------+-----+-------+-----+-------+--------+--------+---------+------+-------+--------------+-----+----------+------+
|Student_ID|school|sex|age|address|famsize|Pstatus|Medu|Fedu|    Mjob|    Fjob|    reason|guardian|traveltime|studytime|failures|schoolsup|famsup|paid|activities|nursery|higher|internet|romantic|famrel|freetime|goout|Dalc|Walc|health|absences| G1| G2| G3|gender|NationalITy|PlaceofBirth|     StageID|GradeID|SectionID|  Topic|Semester|Relation|raisedhands|VisITedResources|AnnouncementsView|Discussion|

Students scoring 15 or above in the final examination are identified as high performers.

In [12]:
from pyspark.sql.functions import avg

education_df.groupBy("sex").agg(
    avg("G3").alias("Average_Final_Marks")).show()

+---+-------------------+
|sex|Average_Final_Marks|
+---+-------------------+
|  F|  9.966346153846153|
|  M| 10.914438502673796|
+---+-------------------+



The average final examination score is calculated separately for male and female students.

In [13]:
education_df.groupBy("status").count().show()

+----------+-----+
|    status|count|
+----------+-----+
|      NULL|  180|
|Not Placed|   67|
|    Placed|  148|
+----------+-----+



This aggregation shows the number of students who are placed and not placed.

In [14]:
education_df.groupBy("Class").agg(
    avg("VisITedResources").alias("Avg_Resources"),
    avg("Discussion").alias("Avg_Discussion")
).show()

+-----+------------------+-----------------+
|Class|     Avg_Resources|   Avg_Discussion|
+-----+------------------+-----------------+
|    M|55.853801169590646| 42.8187134502924|
|    L|18.469026548672566|30.43362831858407|
|    H| 75.50450450450451| 50.8018018018018|
+-----+------------------+-----------------+



This analysis shows how online learning activity varies across different student performance categories.

In [15]:
education_df.groupBy("StageID").agg(
    avg("G3").alias("Average_Final_Marks"),
    avg("raisedhands").alias("Average_Raised_Hands")
).show()

+------------+-------------------+--------------------+
|     StageID|Average_Final_Marks|Average_Raised_Hands|
+------------+-------------------+--------------------+
|  HighSchool|  9.878787878787879|   43.15151515151515|
|MiddleSchool| 10.515337423312884|   48.25153374233129|
|  lowerlevel| 10.422110552763819|  38.447236180904525|
+------------+-------------------+--------------------+



Students are grouped according to their educational stage, and the average academic performance and classroom participation are calculated.

**Q5. Exploratory Data Analysis and Spark SQL**

In [16]:
xapi_df.createOrReplaceTempView("xapi")
student_mat_df.createOrReplaceTempView("student_math")
student_por_df.createOrReplaceTempView("student_por")
placement_df.createOrReplaceTempView("placement")

Temporary views are created so that SQL queries can be executed directly on Spark DataFrames.

In [17]:
#Attendance Distribution
spark.sql("""
SELECT StudentAbsenceDays,
COUNT(*) AS Total_Students
FROM xapi
GROUP BY StudentAbsenceDays
ORDER BY Total_Students DESC
""").show()

+------------------+--------------+
|StudentAbsenceDays|Total_Students|
+------------------+--------------+
|           Under-7|           289|
|           Above-7|           191|
+------------------+--------------+



This query shows how many students fall into each attendance category (Under-7 and Above-7 absence days).

It helps identify the overall attendance pattern.

In [18]:
spark.sql("""
SELECT
AVG(G1) AS Avg_G1,
AVG(G2) AS Avg_G2,
AVG(G3) AS Avg_G3
FROM student_math
""").show()

+-----------------+------------------+------------------+
|           Avg_G1|            Avg_G2|            Avg_G3|
+-----------------+------------------+------------------+
|10.90886075949367|10.713924050632912|10.415189873417722|
+-----------------+------------------+------------------+



The average marks of Mathematics students are calculated for the first, second, and final examinations.

In [19]:
spark.sql("""
SELECT
AVG(G1) AS Avg_G1,
AVG(G2) AS Avg_G2,
AVG(G3) AS Avg_G3
FROM student_por
""").show()

+------------------+------------------+------------------+
|            Avg_G1|            Avg_G2|            Avg_G3|
+------------------+------------------+------------------+
|11.399075500770415|11.570107858243452|11.906009244992296|
+------------------+------------------+------------------+



This query evaluates the academic performance of students in the Portuguese subject.

Comparing both subjects helps determine subject-wise performance.

In [20]:
spark.sql("""
SELECT
Student_ID,
sex,
age,
G3
FROM student_math
ORDER BY G3 DESC
LIMIT 10
""").show()

+----------+---+---+---+
|Student_ID|sex|age| G3|
+----------+---+---+---+
|        47|  M| 16| 20|
|       113|  M| 15| 19|
|         8|  M| 15| 19|
|       110|  M| 15| 19|
|       286|  F| 18| 19|
|       374|  F| 18| 19|
|        91|  F| 15| 18|
|        42|  M| 15| 18|
|        36|  M| 15| 18|
|       104|  M| 15| 18|
+----------+---+---+---+



Students with the highest final examination scores are identified.

These students can be recognized for academic excellence.

In [21]:
spark.sql("""
SELECT
status,
COUNT(*) AS Total_Students
FROM placement
GROUP BY status
""").show()

+----------+--------------+
|    status|Total_Students|
+----------+--------------+
|Not Placed|            67|
|    Placed|           148|
+----------+--------------+



This query shows the number of students who are placed and not placed.

It helps evaluate the university's placement performance.

In [22]:
spark.sql("""
SELECT
AVG(salary) AS Average_Salary
FROM placement
WHERE status='Placed'
""").show()

+-----------------+
|   Average_Salary|
+-----------------+
|288655.4054054054|
+-----------------+



The average salary package offered to placed students is calculated.

This is an important placement KPI.

In [23]:
#Academic Report
spark.sql("""
SELECT
StageID,
GradeID,
COUNT(*) AS Students,
AVG(raisedhands) AS Avg_RaisedHands,
AVG(VisITedResources) AS Avg_Resources,
AVG(Discussion) AS Avg_Discussion
FROM xapi
GROUP BY StageID, GradeID
ORDER BY StageID, GradeID
""").show()

+------------+-------+--------+------------------+------------------+------------------+
|     StageID|GradeID|Students|   Avg_RaisedHands|     Avg_Resources|    Avg_Discussion|
+------------+-------+--------+------------------+------------------+------------------+
|  HighSchool|   G-09|       5|              21.6|              46.0|              58.0|
|  HighSchool|   G-10|       4|             36.75|              44.5|              33.5|
|  HighSchool|   G-11|      13| 58.30769230769231| 62.30769230769231| 55.69230769230769|
|  HighSchool|   G-12|      11| 37.36363636363637|              41.0| 54.90909090909091|
|MiddleSchool|   G-06|      32|            61.625|           54.9375|          45.34375|
|MiddleSchool|   G-07|     100|             48.18|             54.16|             48.47|
|MiddleSchool|   G-08|     116| 56.78448275862069| 63.37068965517241| 43.78448275862069|
|  lowerlevel|   G-02|     147| 36.53061224489796|51.476190476190474|34.374149659863946|
|  lowerlevel|   G-04

This report summarizes student engagement and academic activity across different educational stages and grades.

It serves as an academic performance report similar to a semester-wise report.

In [24]:
#EDA - Gender-wise Performance
student_mat_df.groupBy("sex").avg("G3").show()

+---+------------------+
|sex|           avg(G3)|
+---+------------------+
|  F| 9.966346153846153|
|  M|10.914438502673796|
+---+------------------+



This analysis compares the average final examination marks of male and female students.

In [25]:
#EDA - Parent Education
student_mat_df.groupBy("Medu").count().show()

+----+-----+
|Medu|count|
+----+-----+
|   1|   59|
|   3|   99|
|   4|  131|
|   2|  103|
|   0|    3|
+----+-----+



This shows the distribution of students based on their mother's education level.

It helps analyze whether parental education may influence student performance.

**Key Insights:-**
Attendance analysis identified students with different absence levels.

Subject-wise performance was evaluated using average Mathematics and Portuguese examination scores.

Top-performing students were identified based on final exam marks.

Placement trends were analyzed by counting placed and non-placed students and calculating the average salary of placed students.

Academic reports were generated using educational stages and grades as semester equivalents.

Additional EDA provided insights into gender-wise academic performance and the distribution of students by parental education level.

**Q6. ETL Pipeline Development**

Academic Data ETL Pipeline

In [26]:
academic_raw_df = student_mat_df.select(
    "Student_ID",
    "sex",
    "age",
    "studytime",
    "failures",
    "absences",
    "G1",
    "G2",
    "G3"
)

academic_raw_df.show(5)

+----------+---+---+---------+--------+--------+---+---+---+
|Student_ID|sex|age|studytime|failures|absences| G1| G2| G3|
+----------+---+---+---------+--------+--------+---+---+---+
|         0|  F| 18|        2|       0|       6|  5|  6|  6|
|         1|  F| 17|        2|       0|       4|  5|  5|  6|
|         2|  F| 15|        2|       3|      10|  7|  8| 10|
|         3|  F| 15|        3|       0|       2| 15| 14| 15|
|         4|  F| 16|        2|       0|       4|  6| 10| 10|
+----------+---+---+---------+--------+--------+---+---+---+
only showing top 5 rows


Academic data is extracted from the Mathematics dataset.

Important columns such as study time, failures, absences, and exam scores are selected for further processing.

In [27]:
academic_clean_df = academic_raw_df.dropDuplicates(["Student_ID"])

academic_clean_df = academic_clean_df.fillna({
    "studytime": 0,
    "failures": 0,
    "absences": 0,
    "G1": 0,
    "G2": 0,
    "G3": 0
})

academic_clean_df = academic_clean_df.withColumn(
    "Total_Score",
    col("G1") + col("G2") + col("G3")
)

academic_clean_df = academic_clean_df.withColumn(
    "Average_Score",
    round(col("Total_Score") / 3, 2)
)

academic_clean_df = academic_clean_df.withColumn(
    "Performance_Level",
    when(col("Average_Score") >= 15, "High")
    .when(col("Average_Score") >= 10, "Medium")
    .otherwise("Low")
)

academic_clean_df.show(5)

+----------+---+---+---------+--------+--------+---+---+---+-----------+-------------+-----------------+
|Student_ID|sex|age|studytime|failures|absences| G1| G2| G3|Total_Score|Average_Score|Performance_Level|
+----------+---+---+---------+--------+--------+---+---+---+-----------+-------------+-----------------+
|         0|  F| 18|        2|       0|       6|  5|  6|  6|         17|         5.67|              Low|
|         1|  F| 17|        2|       0|       4|  5|  5|  6|         16|         5.33|              Low|
|         2|  F| 15|        2|       3|      10|  7|  8| 10|         25|         8.33|              Low|
|         3|  F| 15|        3|       0|       2| 15| 14| 15|         44|        14.67|           Medium|
|         4|  F| 16|        2|       0|       4|  6| 10| 10|         26|         8.67|              Low|
+----------+---+---+---------+--------+--------+---+---+---+-----------+-------------+-----------------+
only showing top 5 rows


Academic data is cleaned and transformed.

New features are created:

Total_Score
Average_Score
Performance_Level

These features help in performance analysis and prediction.

In [28]:
academic_clean_df.write.mode("overwrite").parquet(
    "output/academic_etl")

The transformed academic data is loaded into Parquet format.

Parquet is used because it is columnar, compressed, and efficient for Spark analytics.

In [29]:
academic_check_df = spark.read.parquet("output/academic_etl")

academic_check_df.show(5)
print("Academic ETL Records:", academic_check_df.count())

+----------+---+---+---------+--------+--------+---+---+---+-----------+-------------+-----------------+
|Student_ID|sex|age|studytime|failures|absences| G1| G2| G3|Total_Score|Average_Score|Performance_Level|
+----------+---+---+---------+--------+--------+---+---+---+-----------+-------------+-----------------+
|         0|  F| 18|        2|       0|       6|  5|  6|  6|         17|         5.67|              Low|
|         1|  F| 17|        2|       0|       4|  5|  5|  6|         16|         5.33|              Low|
|         2|  F| 15|        2|       3|      10|  7|  8| 10|         25|         8.33|              Low|
|         3|  F| 15|        3|       0|       2| 15| 14| 15|         44|        14.67|           Medium|
|         4|  F| 16|        2|       0|       4|  6| 10| 10|         26|         8.67|              Low|
+----------+---+---+---------+--------+--------+---+---+---+-----------+-------------+-----------------+
only showing top 5 rows
Academic ETL Records: 395


The saved academic ETL output is read back successfully.

This confirms that the academic ETL pipeline executed correctly.

**Placement Data ETL Pipeline**

In [30]:
placement_raw_df = placement_df.select(
    "Student_ID",
    "gender",
    "ssc_p",
    "hsc_p",
    "degree_p",
    "etest_p",
    "mba_p",
    "status",
    "salary"
)

placement_raw_df.show(5)

+----------+------+-----+-----+--------+-------+-----+----------+------+
|Student_ID|gender|ssc_p|hsc_p|degree_p|etest_p|mba_p|    status|salary|
+----------+------+-----+-----+--------+-------+-----+----------+------+
|         0|     M| 67.0| 91.0|    58.0|   55.0| 58.8|    Placed|270000|
|         1|     M|79.33|78.33|   77.48|   86.5|66.28|    Placed|200000|
|         2|     M| 65.0| 68.0|    64.0|   75.0| 57.8|    Placed|250000|
|         3|     M| 56.0| 52.0|    52.0|   66.0|59.43|Not Placed|  NULL|
|         4|     M| 85.8| 73.6|    73.3|   96.8| 55.5|    Placed|425000|
+----------+------+-----+-----+--------+-------+-----+----------+------+
only showing top 5 rows


Placement-related academic and employment features are extracted from the placement dataset.

These columns are useful for placement trend analysis and placement prediction.

In [31]:
placement_clean_df = placement_raw_df.dropDuplicates(["Student_ID"])

placement_clean_df = placement_clean_df.fillna({
    "salary": 0
})

placement_clean_df = placement_clean_df.withColumn(
    "Placement_Flag",
    when(col("status") == "Placed", 1).otherwise(0)
)

placement_clean_df = placement_clean_df.withColumn(
    "Average_Academic_Percentage",
    round(
        (col("ssc_p") + col("hsc_p") + col("degree_p") + col("etest_p") + col("mba_p")) / 5,
        2
    )
)

placement_clean_df = placement_clean_df.withColumn(
    "Package_Category",
    when(col("salary") >= 500000, "High Package")
    .when(col("salary") > 0, "Normal Package")
    .otherwise("Not Placed")
)

placement_clean_df.show(5)

+----------+------+-----+-----+--------+-------+-----+----------+------+--------------+---------------------------+----------------+
|Student_ID|gender|ssc_p|hsc_p|degree_p|etest_p|mba_p|    status|salary|Placement_Flag|Average_Academic_Percentage|Package_Category|
+----------+------+-----+-----+--------+-------+-----+----------+------+--------------+---------------------------+----------------+
|         0|     M| 67.0| 91.0|    58.0|   55.0| 58.8|    Placed|270000|             1|                      65.96|  Normal Package|
|         1|     M|79.33|78.33|   77.48|   86.5|66.28|    Placed|200000|             1|                      77.58|  Normal Package|
|         2|     M| 65.0| 68.0|    64.0|   75.0| 57.8|    Placed|250000|             1|                      65.96|  Normal Package|
|         3|     M| 56.0| 52.0|    52.0|   66.0|59.43|Not Placed|     0|             0|                      57.09|      Not Placed|
|         4|     M| 85.8| 73.6|    73.3|   96.8| 55.5|    Placed|4250

Placement data is cleaned and transformed.

New features are created:

Placement_Flag
Average_Academic_Percentage
Package_Category

These features help analyze placement outcomes and prepare data for ML prediction.

In [32]:
placement_clean_df.write.mode("overwrite").parquet(
    "output/placement_etl"
)

The transformed placement data is saved in Parquet format for further analytics and ML tasks.

In [33]:
placement_check_df = spark.read.parquet("output/placement_etl")

placement_check_df.show(5)
print("Placement ETL Records:", placement_check_df.count())

+----------+------+-----+-----+--------+-------+-----+----------+------+--------------+---------------------------+----------------+
|Student_ID|gender|ssc_p|hsc_p|degree_p|etest_p|mba_p|    status|salary|Placement_Flag|Average_Academic_Percentage|Package_Category|
+----------+------+-----+-----+--------+-------+-----+----------+------+--------------+---------------------------+----------------+
|         0|     M| 67.0| 91.0|    58.0|   55.0| 58.8|    Placed|270000|             1|                      65.96|  Normal Package|
|         1|     M|79.33|78.33|   77.48|   86.5|66.28|    Placed|200000|             1|                      77.58|  Normal Package|
|         2|     M| 65.0| 68.0|    64.0|   75.0| 57.8|    Placed|250000|             1|                      65.96|  Normal Package|
|         3|     M| 56.0| 52.0|    52.0|   66.0|59.43|Not Placed|     0|             0|                      57.09|      Not Placed|
|         4|     M| 85.8| 73.6|    73.3|   96.8| 55.5|    Placed|4250

The placement ETL output is successfully loaded again from storage.

This confirms that the placement ETL pipeline is working properly.

Academic ETL Pipeline
Extract:
Selected academic columns from student-mat dataset.

Transform:
Removed duplicates, handled missing values, calculated total score, average score, and performance level.

Load:
Saved transformed academic data in Parquet format.


Placement ETL Pipeline
Extract:
Selected placement-related columns from placement dataset.

Transform:
Removed duplicates, handled missing salary values, created placement flag, academic percentage average, and package category.

Load:
Saved transformed placement data in Parquet format.

**Q7. Machine Learning/Deep Learning Implementation**

In [34]:
from pyspark.sql.functions import when

ml_df = placement_df.select(
    "ssc_p",
    "hsc_p",
    "degree_p",
    "etest_p",
    "mba_p",
    "status" )

ml_df.show(5)

+-----+-----+--------+-------+-----+----------+
|ssc_p|hsc_p|degree_p|etest_p|mba_p|    status|
+-----+-----+--------+-------+-----+----------+
| 67.0| 91.0|    58.0|   55.0| 58.8|    Placed|
|79.33|78.33|   77.48|   86.5|66.28|    Placed|
| 65.0| 68.0|    64.0|   75.0| 57.8|    Placed|
| 56.0| 52.0|    52.0|   66.0|59.43|Not Placed|
| 85.8| 73.6|    73.3|   96.8| 55.5|    Placed|
+-----+-----+--------+-------+-----+----------+
only showing top 5 rows


The important academic percentage columns and the placement status are selected for building the machine learning model.

In [35]:
ml_df = ml_df.withColumn(
    "label",
    when(ml_df.status == "Placed", 1).otherwise(0)
)

ml_df.show(5)

+-----+-----+--------+-------+-----+----------+-----+
|ssc_p|hsc_p|degree_p|etest_p|mba_p|    status|label|
+-----+-----+--------+-------+-----+----------+-----+
| 67.0| 91.0|    58.0|   55.0| 58.8|    Placed|    1|
|79.33|78.33|   77.48|   86.5|66.28|    Placed|    1|
| 65.0| 68.0|    64.0|   75.0| 57.8|    Placed|    1|
| 56.0| 52.0|    52.0|   66.0|59.43|Not Placed|    0|
| 85.8| 73.6|    73.3|   96.8| 55.5|    Placed|    1|
+-----+-----+--------+-------+-----+----------+-----+
only showing top 5 rows


Machine learning algorithms require numeric target values.

Placed → 1
Not Placed → 0

In [36]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=[
        "ssc_p",
        "hsc_p",
        "degree_p",
        "etest_p",
        "mba_p"
    ],
    outputCol="features"
)

data = assembler.transform(ml_df)

data.select("features", "label").show(5, truncate=False)

+------------------------------+-----+
|features                      |label|
+------------------------------+-----+
|[67.0,91.0,58.0,55.0,58.8]    |1    |
|[79.33,78.33,77.48,86.5,66.28]|1    |
|[65.0,68.0,64.0,75.0,57.8]    |1    |
|[56.0,52.0,52.0,66.0,59.43]   |0    |
|[85.8,73.6,73.3,96.8,55.5]    |1    |
+------------------------------+-----+
only showing top 5 rows


The selected academic features are combined into a single features vector, which is the required input format for Spark ML algorithms.

In [37]:
train_data, test_data = data.randomSplit([0.8, 0.2], seed=42)

print("Training Records:", train_data.count())
print("Testing Records:", test_data.count())

Training Records: 182
Testing Records: 33


The dataset is divided into:

80% Training Data
20% Testing Data

This allows the model to learn from one dataset and evaluate on unseen data.

In [38]:
from pyspark.ml.classification import LogisticRegression

lr = LogisticRegression(
    featuresCol="features",
    labelCol="label"
)

model = lr.fit(train_data)

A Logistic Regression model is trained to learn the relationship between students' academic performance and their placement status.

In [39]:
predictions = model.transform(test_data)

predictions.select(
    "features",
    "label",
    "prediction",
    "probability"
).show(10, truncate=False)

+-----------------------------+-----+----------+-----------------------------------------+
|features                     |label|prediction|probability                              |
+-----------------------------+-----+----------+-----------------------------------------+
|[43.0,60.0,65.0,92.66,62.92] |0    |0.0       |[0.9823700337963067,0.01762996620369328] |
|[47.0,55.0,65.0,62.0,65.04]  |0    |0.0       |[0.9865300461813793,0.013469953818620706]|
|[48.0,51.0,58.0,60.0,58.79]  |0    |0.0       |[0.9863430931274944,0.013656906872505647]|
|[51.0,54.0,61.0,60.0,60.64]  |0    |0.0       |[0.9650801510323141,0.03491984896768585] |
|[52.0,57.0,50.8,67.0,62.79]  |0    |0.0       |[0.9912407268279064,0.008759273172093573]|
|[52.0,65.0,57.0,75.0,59.81]  |0    |0.0       |[0.9087479524799497,0.09125204752005034] |
|[54.0,82.0,63.0,50.0,59.47]  |0    |1.0       |[0.28900614712797834,0.7109938528720217] |
|[55.68,61.33,56.87,66.0,58.3]|1    |0.0       |[0.8486210571473841,0.15137894285261588] |

The trained model predicts:

Actual Label
Predicted Label
Probability of Placement

This helps estimate the placement probability for each student.

In [40]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator

evaluator = BinaryClassificationEvaluator(
    labelCol="label"
)

accuracy = evaluator.evaluate(predictions)

print("Model Accuracy (AUC):", accuracy)

Model Accuracy (AUC): 0.9022556390977444


The Binary Classification Evaluator calculates the Area Under the ROC Curve (AUC).

A higher AUC value indicates better classification performance

In [41]:
predictions.select(
    "ssc_p",
    "hsc_p",
    "degree_p",
    "mba_p",
    "prediction"
).show(10)

+-----+-----+--------+-----+----------+
|ssc_p|hsc_p|degree_p|mba_p|prediction|
+-----+-----+--------+-----+----------+
| 43.0| 60.0|    65.0|62.92|       0.0|
| 47.0| 55.0|    65.0|65.04|       0.0|
| 48.0| 51.0|    58.0|58.79|       0.0|
| 51.0| 54.0|    61.0|60.64|       0.0|
| 52.0| 57.0|    50.8|62.79|       0.0|
| 52.0| 65.0|    57.0|59.81|       0.0|
| 54.0| 82.0|    63.0|59.47|       1.0|
|55.68|61.33|   56.87| 58.3|       0.0|
| 59.0| 60.0|    56.0| 57.9|       0.0|
| 59.0| 62.0|    77.5| 67.0|       1.0|
+-----+-----+--------+-----+----------+
only showing top 10 rows


The model predicts whether each student is likely to be Placed or Not Placed based on academic performance.

Outcomes:-

Academic features were converted into machine learning feature vectors.

A Logistic Regression model was trained using Spark MLlib.

The model predicted whether a stud ent would be Placed or Not Placed.

Performance was evaluated using AUC (Area Under the ROC Curve).

This demonstrates how Apache Spark can be used to build scalable machine learning pipelines for educational analytics and placement prediction.